In [1]:
# krylov complexity codes
#  from scipy.sparse import csr_matrix

# # 1. Make K vectors
# def get_K(n_sites, H, psi_0, max_iter=None):
#     """
#     Calculates the K vectors K_n for evolution under a given Hamiltonian.
#     We must change Hamiltonian and psi_0 to get k vectors for different systems
#     """
#     # Convert initial state to a flat numpy array
#     v_curr = np.array(psi_0.full(), dtype=complex).flatten()
#     v_curr = v_curr / np.linalg.norm(v_curr)
#     H_sparse = csr_matrix(H.full(), dtype = complex)
#     # H_sparse = H.data # qutips inbuilt sparse format output
    
#     dim = v_curr.shape[0]
#     if max_iter is None:
#         max_iter = dim

#     alpha_list = []
#     beta_list = []
    
#     v_prev = np.zeros(dim, dtype=complex)
#     beta = 0.0
    
#     for i in range(max_iter):
#         # Lanczos step: w = H|v_n>
#         w = H_sparse @ v_curr
        
#         # alpha_n = <v_n|H|v_n>
#         alpha = np.vdot(v_curr, w)
#         alpha_list.append(alpha)
        
#         # Orthogonalize: w_next = H|v_n> - alpha_n|v_n> - beta_n|v_{n-1}>
#         w = w - alpha * v_curr - beta * v_prev 
#         beta = np.linalg.norm(w)
#         if beta < 1e-12:
#             print(f"Krylov subspace exhausted at step {i+1}")
#             break
        
#         beta_list.append(beta)
        
#         # Update vectors for next iteration
#         v_prev = v_curr
#         v_curr = w / beta
            
#     return np.array(beta_list), np.array(alpha_list)

In [2]:
# one with full re orthogonalisation 

from scipy.sparse import csr_matrix

# 1. Make K vectors
def get_K(n_sites, H, psi_0, max_iter=None):
    """
    Calculates the K vectors K_n for evolution under a given Hamiltonian.
    We must change Hamiltonian and psi_0 to get k vectors for different systems
    """
    # Convert initial state to a flat numpy array
    v_curr = np.array(psi_0.full(), dtype=complex).flatten()   # we will do something if initial state is not Qobj
    # v_curr = np.array(psi_0)
    v_curr = v_curr / np.linalg.norm(v_curr)
    # H_sparse = csr_matrix(H.full(), dtype = complex)    # same, we have to do something is H is not Qobj
    H_sparse = csr_matrix(H, dtype=complex)
    # H_sparse = H.data # qutips inbuilt sparse format output
    
    dim = v_curr.shape[0]
    if max_iter is None:
        max_iter = dim

    alpha_list = []
    beta_list = []
    store_k = []
    v_prev = np.zeros(dim, dtype=complex)
    beta = 0.0
    
    for i in range(max_iter):
        # Lanczos step: w = H|v_n>
        store_k.append(v_curr.copy())
        w = H_sparse @ v_curr
        
        # alpha_n = <v_n|H|v_n>
        alpha = np.vdot(v_curr, w)
        alpha_list.append(alpha)
        
        # Orthogonalize: w_next = H|v_n> - alpha_n|v_n> - beta_n|v_{n-1}>
        w = w - alpha * v_curr - beta * v_prev 
        for v_k in store_k:
            w -= np.vdot(v_k, w) * v_k
            
        beta = np.linalg.norm(w)
        if beta < 1e-12:
            print(f"Krylov subspace exhausted at step {i+1}")
            break
        
        beta_list.append(beta)
        
        # Update vectors for next iteration
        v_prev = v_curr
        v_curr = w / beta
            
    return np.array(beta_list), np.array(alpha_list)

In [3]:
def get_T(alphas, betas):
    """
    makes T matrix from alpha and beta and not matrix miltiplication
    as it is still computationally expensive and we dont get sparse matrices 
    where most terms are zeros
    """
    K = len(alphas)
    T = np.diag(alphas).astype(complex)
    if len(betas) > 0:
        # We use betas[:K-1] to ensure dimensions match
        T += np.diag(betas[:K-1], k=1)
        T += np.diag(betas[:K-1], k=-1)
    return T

In [4]:
# using T matrix we significatly cut the computational cost

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import expm_multiply
import matplotlib.pyplot as plt
from qutip import tensor

from scipy.sparse import diags
from scipy.sparse.linalg import expm_multiply

def get_krylov_overlaps(T_matrix, t_max=200, steps=400):
    
    T_sparse = csr_matrix(T_matrix)
    dim = T_matrix.shape[0]
    # Initial state in Krylov basis is ALWAYS [1, 0, 0, ...]
    psi_krylov = np.zeros(dim, dtype=complex)
    psi_krylov[0] = 1.0 
    
    overlaps = np.zeros((dim, steps), dtype=complex)
    dt = t_max / (steps - 1)
    
    current_psi = psi_krylov
    for t_idx in range(steps):
        # These are the amplitudes <K_n | psi(t)>
        overlaps[:, t_idx] = current_psi
        
        if t_idx < steps - 1:
            # Evolve in the Krylov subspace
            current_psi = expm_multiply(-1j * T_sparse * dt, current_psi)
            
    return overlaps
